In [ ]:
import pandas as pd
import numpy as np
import warnings
from IPython.display import display

warnings.filterwarnings('ignore')
seed = 21
path = '../input/zindi-new-user-engagement/'

In [ ]:
CompetitionPartipation = pd.read_csv(path + "CompetitionPartipation.csv")
UserActivity = pd.read_csv(path + "UserActivity.csv")
Competition = pd.read_csv(path + "Competition.csv")
Blogs = pd.read_csv(path + "Blogs.csv")
SampleSubmission = pd.read_csv(path + "SampleSubmission.csv")
VariableDefinitions = pd.read_csv(path + "VariableDefinitions.csv")
Users = pd.read_csv(path + "Users.csv")
Discussion = pd.read_csv(path + "Discussion.csv")
Jobs = pd.read_csv(path + "Users.csv")
Comments = pd.read_csv(path + "Comments.csv")

display(
#     "CompetitionPartipation", CompetitionPartipation,
    "UserActivity", UserActivity,
    "Users", Users, 
    # "Competition", Competition,
    # "Blogs", Blogs, 
    # "Jobs", Jobs,
    # "Discussion", Discussion, 
    # "Comments", Comments,
    # "VariableDefinitions", VariableDefinitions, 
    "SampleSubmission", SampleSubmission, 
)

In [ ]:
UserActivity.describe(include='all')

In [ ]:
UserActivity['datetime Month'].value_counts()

there are 7 months that are given on the activity set

In [ ]:
set(UserActivity[UserActivity['datetime Month'] == 5].User_ID.unique()).\
    intersection(set(SampleSubmission.User_ID_Next_month_Activity.str.replace('_Month_5', '')))

since there is no intersection between the submission file ids and the user activities ids, we need to predict the user activities for the ids that are on the samplesubmission file

In [ ]:
UserActivity.Title.value_counts()

In [ ]:
# lets reduce the types of activities by removing the ids from them
def remove_id(x):
    x = x.replace('$', '')
    if 'ID' in x:
        return "_".join(x.split('_')[:-1])
    
    return x.replace(' ', '_')

UserActivity.Title.apply(remove_id).value_counts()

In [ ]:
UserActivity['main_activities'] = UserActivity.Title.apply(remove_id)
UserActivity.main_activities.unique()

Let's select the activities that are done usually

In [ ]:
act_vc = UserActivity.main_activities.value_counts()
selected_activities = act_vc[act_vc > 1000].keys()
selected_activities

In [ ]:
UserActivity = UserActivity[UserActivity.main_activities.isin(selected_activities)].reset_index(drop=True)
UserActivity

In [ ]:
# aggregate the number of times that a user has taken per month for each type of activities
dataset = UserActivity.groupby(['User_ID', 'main_activities', 'datetime Month']).size().reset_index().\
    rename(columns={0: 'count'}).\
        sort_values(['User_ID', 'datetime Month', 'count'], ascending=False)

dataset    

We can have a model to train on the user activities on 

In [ ]:
dataset.User_ID.nunique(), dataset[dataset['datetime Month'] == 5].User_ID.nunique(),  SampleSubmission.User_ID_Next_month_Activity.nunique()

Only for the 3.5k users the information is required but to have more data we need to include users that have activities on past months

In [ ]:
dataset.main_activities.nunique()

we will build a model to predict the number of time that a user will take any of the 22 actions and from that we'll predict the user's engagement

In [ ]:
Selected_users = dataset[['User_ID']].drop_duplicates().reset_index(drop=True)
# add ids from SampleSubmission file that are not in the Selected users
# Selected_users = Selected_users.append(pd.DataFrame(SampleSubmission.User_ID_Next_month_Activity.str.replace('_Month_5', '').unique(), columns=['User_ID']), ignore_index=True)
Selected_users

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

Users.Countries_ID = Users.Countries_ID.fillna('unspecified')
Users.Countries_ID = le.fit_transform(Users.Countries_ID)
user_df = pd.merge(Selected_users, Users, on='User_ID',  how='inner')
user_df.drop(columns=['Created At time', 'Created At Year'], inplace=True)
user_df

In [ ]:
all_activities = dataset.main_activities.unique()
month_cols = [f'datetime Month_{i}' for i in [1, 2, 3, 4, 5, 11, 12]]
test_ids = SampleSubmission.User_ID_Next_month_Activity.str.replace('_Month_5', '').unique()

activity_df = user_df[['User_ID']]

for activity in all_activities:
    _df = pd.get_dummies(
        dataset[dataset.main_activities == activity].drop(columns='main_activities').reset_index(drop=True),
        columns=['datetime Month'])
    for col in month_cols:
        if col not in _df.columns:
            _df[col] = 0
        _df[col] *= _df['count']
    _df = _df.drop(columns='count')
    # accumulate the counts
    _df = _df.groupby('User_ID').sum().reset_index()
    # merge with all users
    _df = pd.merge(user_df[['User_ID']], _df, on='User_ID', how='left')
    # fill missing values with 0
    _df[month_cols] = _df[month_cols].fillna(0)
    _df.columns = _df.columns.str.replace('datetime Month_', f'{activity}_month_')

    activity_df = pd.merge(activity_df, _df, on='User_ID', how='left')

month_5 = activity_df.columns[activity_df.columns.str.endswith('_month_5')]
activity_df['Active'] = activity_df[month_5].sum(axis=1)
activity_df['Active'] = activity_df['Active'].apply(lambda x: 1 if x > 0 else 0)
activity_df.drop(columns=month_5, inplace=True)

# merge with users
activity_df = pd.merge(activity_df, user_df, on='User_ID', how='left')

train_df = activity_df[~activity_df.User_ID.isin(test_ids)].reset_index(drop=True)
test_df = activity_df[activity_df.User_ID.isin(test_ids)].drop(columns='Active').reset_index(drop=True)

display(train_df, test_df)

In [ ]:
train_df.Active.value_counts()

now we have created the train and test set, lets go for adversarial validation

In [ ]:
def check_similarity(df1, df2, train_columns):
  from sklearn.metrics import accuracy_score,roc_auc_score,f1_score
  from sklearn.model_selection import train_test_split
  from sklearn.ensemble import RandomForestClassifier
  import matplotlib.pyplot as plt

  d1 = df1.copy()
  d2 = df2.copy()
  d1['source'] = 0
  d2['source'] = 1
  joint_df = pd.concat([d1,d2])
  X = joint_df[train_columns]
  y = joint_df['source']
  print(y.value_counts())
  train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=12, shuffle=True)
  model= RandomForestClassifier(max_depth=15, n_estimators=100, random_state=12)
  model.fit(train_X,train_y)
  print('accuracy is {}'.format(accuracy_score(model.predict(test_X),test_y)))
  print('f1 score  is {}'.format(f1_score(model.predict(test_X),test_y)))
  feat_importances = pd.Series(model.feature_importances_, index=X.columns)
  feat_importances.nlargest(20).plot(kind='barh')
  plt.show()

check_similarity(train_df, test_df, train_df.columns.difference([
  'User_ID', 'Active', 'Signed_Up_month_4', 'badge_OCZE_month_4', 'create_alias_month_4',
  'Confirmed_Email_month_4', 'Viewed_All_Discussions_month_4', 'Viewed_All_Competitions_month_4',
  'Viewed_All_Learning_Pages_month_4', 'Updated_Profile_month_4', 'identify_month_4', 'comp_ID_month_4',
  'Joined_Competition_month_4', 'Downloaded_Competition_Datafile_month_4', 'Created At Month',
]))

In [ ]:
main_cols = train_df.columns.difference([
    'User_ID', 'Active', 'Signed_Up_month_4', 'badge_OCZE_month_4', 'create_alias_month_4',
    'Confirmed_Email_month_4', 'Viewed_All_Discussions_month_4', 'Viewed_All_Competitions_month_4',
    'Viewed_All_Learning_Pages_month_4', 'Updated_Profile_month_4', 'identify_month_4', 'comp_ID_month_4',
    'Joined_Competition_month_4', 'Downloaded_Competition_Datafile_month_4', 'Created At Month'
])
X = train_df[main_cols]
y = train_df['Active']
test = test_df[main_cols]

In [ ]:
# train a model to predict the active users

from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import f1_score


model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.06124,
)

skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
oof = np.zeros(len(train_df))
predictions = []

for fold, (trn_idx, val_idx) in enumerate(skfold.split(X, y)):
    print(f'Fold {fold + 1}')
    X_train, y_train = X.iloc[trn_idx], y.iloc[trn_idx]
    X_valid, y_valid = X.iloc[val_idx], y.iloc[val_idx]
    
    model.fit(
        X_train, y_train,
        eval_set=(X_valid, y_valid),
        verbose=200,
        early_stopping_rounds=100,
    )

    oof[val_idx] = model.predict(X_valid)
    predictions.append(model.predict_proba(test)[:, 1])

predictions = np.mean(predictions, axis=0)
print(f'Our oof f1 score is {f1_score(y, oof)}')

In [ ]:
# plot model feature importance

import matplotlib.pyplot as plt
import seaborn as sns

feat_importances = pd.Series(model.feature_importances_, index=X.columns)
feat_importances.nlargest(20).plot(kind='barh')
plt.show()

In [ ]:
sub = pd.DataFrame({
    'User_ID_Next_month_Activity': test_df.User_ID,
    'Active': predictions
})
sub.Active.describe()

In [ ]:
sub.Active = sub.Active.apply(lambda x: 1 if x > 0.96 else 0)
sub.User_ID_Next_month_Activity = sub.User_ID_Next_month_Activity + '_Month_5'
sub

In [ ]:
sub = pd.merge(SampleSubmission.drop(columns='Active'), sub, on='User_ID_Next_month_Activity', how='left').fillna(0)
sub.Active.value_counts()

In [ ]:
sub.to_csv('submission.csv', index=False)

Currently the score is 0.305170239 at public leaderboard<br>
Things to add:- 
 - Use IDs that are on submission file but not on the user activities
 - Use more activity types
 - Use the other csv file to add more information about the activities
 - Use more models

 